# JEV Agent: Ollama на Colab T4 из VS Code

Открой этот notebook в VS Code и выбери **Select Kernel → Colab → New Colab Server**, затем среду с GPU T4. Все ячейки выполняются на Colab, обычный терминал VS Code может оставаться локальным. Ollama и программа запускаются в одной удалённой среде, поэтому туннель не нужен.

Для первого технического запуска здесь явно выбрана `qwen2.5:7b`: обычная instruct-модель с поддержкой русского языка и JSON. Это начальная модель для проверки подключения, а не утверждение о лучшей точности. Ранее обсуждавшуюся `qwen3:8b` можно выбрать переменной `MODEL`; её режим thinking может увеличить время и расход выходных токенов.

Выполняй ячейки по порядку. Установка Ollama и скачивание модели потребуют времени. После сброса Colab VM установку и загрузку файлов нужно повторить.

Документация: [Colab в VS Code](https://github.com/googlecolab/colab-vscode/wiki/User-Guide), [установка Ollama](https://docs.ollama.com/linux), [локальный API](https://docs.ollama.com/api/openai-compatibility), [Qwen2.5 7B](https://ollama.com/library/qwen2.5:7b).

## 1. Проверить GPU
В выводе должно быть название выделенной GPU. Если GPU нет, выбери Colab-сервер с ускорителем. Память GPU уже может быть занята другими моделями в этой VM.

In [2]:
import json
import os
import shutil
import subprocess
import sys
import time
from getpass import getpass
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

def run_command(command, *, cwd=None):
    # Вывод дочернего процесса явно отправляется в ячейку VS Code.
    with subprocess.Popen(
        command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    if code != 0:
        raise RuntimeError(f'Команда завершилась с кодом {code}; подробности выше.')

MODEL = 'qwen2.5:7b'
OLLAMA_URL = 'http://127.0.0.1:11434'
run_command(['nvidia-smi'])


Thu Oct 15 07:12:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Установить Ollama
Сначала устанавливается `zstd`, если его нет: актуальному архиву Ollama нужен этот распаковщик. Затем отдельно скачивается и запускается официальный установщик. Вывод каждой команды виден в ячейке. В Colab может появиться предупреждение об отсутствии systemd; сервер запустим вручную следующей ячейкой. Если команда завершилась ошибкой, смотри сообщение над traceback.

In [3]:
if shutil.which('zstd') is None:
    install_prefix = [] if os.geteuid() == 0 else ['sudo']
    if install_prefix and shutil.which('sudo') is None:
        raise RuntimeError('Для установки zstd нужны права root или sudo')
    print('Устанавливаем zstd...', flush=True)
    run_command(install_prefix + ['apt-get', 'update', '-qq'])
    run_command(install_prefix + ['apt-get', 'install', '-y', 'zstd'])

print('Скачиваем установщик Ollama...', flush=True)
run_command(['curl', '-fSL', '--retry', '2',
             'https://ollama.com/install.sh', '-o', '/tmp/jev-ollama-install.sh'])
run_command(['sh', '/tmp/jev-ollama-install.sh'])


Скачиваем установщик Ollama...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

100 15902  100 15902    0     0  30462      0 --:--:-- --:--:-- --:--:-- 30462
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
#=#=#                                                                          
##O#-#                                                                         

                                                                           0.8%
##                                     

## 3. Запустить сервер в фоне
Сервер слушает только loopback Colab VM. При повторном запуске ячейки уже работающая Ollama переиспользуется. Лог нового процесса: `/tmp/jev-ollama.log`.

In [4]:
def ollama_ready():
    try:
        with urlopen(OLLAMA_URL + '/api/version', timeout=1) as response:
            return response.status == 200 and 'version' in json.load(response)
    except (URLError, OSError, ValueError):
        return False

if not ollama_ready():
    ollama_env = os.environ.copy()
    ollama_env.update({
        'OLLAMA_HOST': '127.0.0.1:11434',
        'OLLAMA_CONTEXT_LENGTH': '8192',
        'OLLAMA_NUM_PARALLEL': '1',
    })
    with open('/tmp/jev-ollama.log', 'ab') as log:
        ollama_process = subprocess.Popen(
            ['ollama', 'serve'], env=ollama_env,
            stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
        )
    for _ in range(30):
        if ollama_ready():
            break
        if ollama_process.poll() is not None:
            raise RuntimeError('Ollama завершилась при запуске; проверь /tmp/jev-ollama.log')
        time.sleep(0.5)
    else:
        raise RuntimeError('Ollama не запустилась; проверь /tmp/jev-ollama.log')

print('Ollama доступна:', OLLAMA_URL)


Ollama доступна: http://127.0.0.1:11434


## 4. Скачать модель и проверить JSON-ответ
Это пробный запрос без Jev. Он загружает модель в память и проверяет тот же `/v1/chat/completions`, который использует проект. После ответа `ollama ps` показывает использование GPU: ожидается `100% GPU` либо GPU-часть при смешанном размещении.

In [5]:
run_command(['ollama', 'pull', MODEL])
payload = {
    'model': MODEL,
    'messages': [{'role': 'user', 'content': 'Верни только JSON: {"ready": true}'}],
    'response_format': {'type': 'json_schema', 'json_schema': {
        'name': 'readiness', 'strict': True, 'schema': {
            'type': 'object', 'properties': {'ready': {'type': 'boolean', 'const': True}},
            'required': ['ready'], 'additionalProperties': False,
        },
    }},
    'temperature': 0,
    'max_tokens': 256,
}
request = Request(
    OLLAMA_URL + '/v1/chat/completions',
    data=json.dumps(payload).encode('utf-8'),
    headers={'Content-Type': 'application/json'}, method='POST',
)
with urlopen(request, timeout=300) as response:
    data = json.load(response)
choice = data['choices'][0]
if choice.get('finish_reason') != 'stop':
    raise RuntimeError('Модель не завершила пробный ответ; проверь лимит токенов и режим thinking')
answer = json.loads(choice['message']['content'])
if answer != {'ready': True}:
    raise RuntimeError('Модель не вернула ожидаемый JSON')
print('JSON Schema API работает, модель:', MODEL)
run_command(['ollama', 'ps'])


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 2bada8a74506: 100% ▕██████████████████▏ 4.7 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕██████████████████▏ 1.5 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling 2f15b3218f05: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 
JSON API работает, модель: qwen2.5:7b
NAME          ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
qwen2.5:7b    845dbda0ea48    5.1 GB    100% GPU     8192       llamacpp    4 minutes from now    


## 5. Перенести текущий проект из VS Code
В локальном Explorer выдели **src**, **data**, **pyproject.toml**, **uv.lock**, **README.md** из папки `jev-agent`. Правой кнопкой → **Upload to Colab** → выбери тот же сервер, что используется этим notebook. Передавай текущие файлы, включая ещё не закоммиченные изменения. `.venv`, `.git` и `.env` не нужны. Ключ Jev введём отдельно.

Файлы должны оказаться в `/content`: проверь дерево **Colab → Contents**. Если ты загрузил папку проекта целиком и она оказалась в `/content/jev-agent`, ячейка ниже также найдёт её. [Инструкция расширения](https://github.com/googlecolab/colab-vscode/wiki/User-Guide#uploading-files-to-colab-servers).

In [6]:
PROJECT_DIR = Path('/content')
if not (PROJECT_DIR / 'pyproject.toml').is_file():
    PROJECT_DIR = Path('/content/jev-agent')
required = ['pyproject.toml', 'uv.lock', 'src/main.py',
            'data/cases.jsonl', 'data/labels.jsonl', 'data/form_templates.json']
missing = [name for name in required if not (PROJECT_DIR / name).is_file()]
if missing:
    raise RuntimeError('Загрузи файлы проекта в Colab; отсутствуют: ' + ', '.join(missing))
print('Проект найден:', PROJECT_DIR)

# Сам notebook остаётся на Python Colab. Программа использует отдельный Python 3.14.
run_command([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
UV_BIN = shutil.which('uv')
if UV_BIN is None:
    raise RuntimeError('uv не найден после установки')
run_command([UV_BIN, 'python', 'install', '3.14'])
run_command([UV_BIN, 'sync', '--locked', '--no-dev', '--python', '3.14'],
            cwd=PROJECT_DIR)


Проект найден: /content
Python 3.14 is already installed
Resolved 23 packages in 1ms
Checked 15 packages in 0.43ms


## 6. Настройки и ключ Jev
Ключ вводится через скрытый ввод и хранится в окружении процесса. Не вставляй его в исходник notebook. Ollama использует заглушку `ollama`; ключ провайдера LLM не нужен. Jev по-прежнему вызывается через свой API.

Включены `LLM_RESPONSE_FORMAT=json_schema` и `LLM_TEMPERATURE=0`: сервер получает схему, а не только просьбу вернуть JSON. Если сервер отклонит схему, программа явно сообщит об ошибке, без скрытого переключения режима. Проверки источников, формул и смысла остаются обязательными.

Тарифы локальной LLM равны нулю для оценки оплаты API; вычисления Colab и расход Jev сюда не включены. После сброса kernel настройки и ключ нужно задать снова.

In [8]:
os.environ.update({
    'LLM_API_KEY': 'ollama',
    'LLM_API_PROTOCOL': 'openai',
    'LLM_MODEL': MODEL,
    'LLM_BASE_URL': OLLAMA_URL + '/v1',
    'LLM_TOKEN_LIMIT_PARAMETER': 'max_tokens',
    'LLM_JSON_MODE': 'true',
    'LLM_RESPONSE_FORMAT': 'json_schema',
    'LLM_TEMPERATURE': '0',
    'LLM_TIMEOUT_SECONDS': '180',
    'LLM_MAX_OUTPUT_TOKENS': '4096',
    'LLM_MAX_RETRIES': '1',
    'LLM_MIN_REQUEST_INTERVAL_SECONDS': '0',
    'LLM_RATE_LIMIT_RETRIES': '1',
    'LLM_INPUT_PRICE_PER_MILLION': '0',
    'LLM_OUTPUT_PRICE_PER_MILLION': '0',
    'EXTRACTION_MAX_ATTEMPTS': '2',
    'JEV_WRONG_THRESHOLD': '0.7',
})
if not os.environ.get('JEV_API_KEY', '').strip():
    os.environ['JEV_API_KEY'] = getpass('Введи JEV_API_KEY (скрытый ввод): ').strip()
if not os.environ['JEV_API_KEY']:
    raise RuntimeError('Нужен непустой JEV_API_KEY')
print('Настройки заданы; ключ не выводится')


Настройки заданы; ключ не выводится


## 7. Запустить один dev-кейс
Эта ячейка выполняет реальные API-запросы к Jev и локальной Ollama. Сначала оставь `CASE_LIMIT = 1`. После успешного запуска и просмотра результата поменяй его на `None`, чтобы обработать весь dev. `test` оставь для итоговой оценки.
Ошибки кейсов не прерывают полный прогон: успешные результаты сохраняются в `outputs/predictions.jsonl`, причины ошибок — в `outputs/predictions.errors.jsonl`, после каждого кейса. Новый прогон заменяет предыдущий снимок; автоматического возобновления нет. В итоговой сводке проверь число ошибок. Для остановки на первой ошибке добавь `--fail-fast` в команду.

`DEBUG = True` включает вывод эталона из `labels.jsonl`, исходного ответа LLM на каждой попытке, этапов и замечаний Python/Jev, а также итогового результата при успехе. Эталон не передаётся модели. Вывод может содержать реквизиты учебного диалога; не публикуй его для реальных данных. Для обычного прогона поставь `DEBUG = False`. Если VS Code сворачивает длинный вывод, открой его через **Open in a text editor**.

Программа запускается дочерним процессом через `uv`, поэтому существующий цикл событий notebook не мешает `asyncio.run()` в `main`.

In [19]:
CASE_LIMIT = None  # None — все dev-кейсы
DEBUG = True  # Ответ LLM, эталон и замечания; вывод может содержать реквизиты
command = [UV_BIN, 'run', '--locked', '--no-dev', '--python', '3.14',
           'python', '-m', 'src.main', '--mode', 'full', '--split', 'dev']
if CASE_LIMIT is not None:
    command.extend(['--limit', str(CASE_LIMIT)])
if DEBUG:
    command.append('--debug')
run_command(command, cwd=PROJECT_DIR)
print('Прогон завершён')


DEBUG: ответы модели и эталоны могут содержать реквизиты; не публикуйте вывод.
[1/28] C001
[C001] Ожидаемый результат из labels.jsonl (не передаётся модели):
{
  "case_id": "C001",
  "document_type": "invoice_draft",
  "fields": {
    "recipient_name": {
      "status": "fillable",
      "value": "ООО Ромашка",
      "evidence": [
        "m1"
      ],
      "derivation": null
    },
    "recipient_inn": {
      "status": "fillable",
      "value": "7701234567",
      "evidence": [
        "m1"
      ],
      "derivation": null
    },
    "recipient_email": {
      "status": "missing",
      "value": null,
      "evidence": [],
      "derivation": null
    },
    "recipient_account": {
      "status": "missing",
      "value": null,
      "evidence": [],
      "derivation": null
    },
    "recipient_bik": {
      "status": "missing",
      "value": null,
      "evidence": [],
      "derivation": null
    },
    "subject": {
      "status": "fillable",
      "value": "консультационные 

## 8. Посмотреть и сохранить результат
Успешные результаты находятся в Colab по пути `PROJECT_DIR/outputs/predictions.jsonl`, ошибки — в `PROJECT_DIR/outputs/predictions.errors.jsonl`. Ячейка показывает первую успешную запись и список ошибок; при отсутствии успешных результатов файл предсказаний пустой. Скопируй оба файла в локальный проект через файловую панель Colab до завершения VM. Локальные файлы VS Code и файлы Colab — разные копии: после правок кода загрузи обновлённые файлы ещё раз.

In [ ]:
output_path = PROJECT_DIR / 'outputs' / 'predictions.jsonl'
if not output_path.is_file():
    raise RuntimeError('Файл результата отсутствует; сначала выполни прогон')
with output_path.open(encoding='utf-8') as output_file:
    first_line = output_file.readline()
print('Файл:', output_path)
if first_line.strip():
    print(json.dumps(json.loads(first_line), ensure_ascii=False, indent=2))
else:
    print('Нет успешных результатов; проверь файл ошибок')
errors_path = output_path.with_name(output_path.stem + '.errors.jsonl')
if errors_path.is_file():
    print('Файл ошибок:', errors_path)
    with errors_path.open(encoding='utf-8') as errors_file:
        for line in errors_file:
            if line.strip():
                error = json.loads(line)
                print(error['case_id'], ':', error['error'])
